In [19]:
import sys, os, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, root_mean_squared_error
sys.path.append(os.path.abspath('..'))
from scripts.pipeline import data_engineering


In [20]:
# The trained spline model is pulled to use in test and to not allow data leakage
train_spline = joblib.load('../models/training_spline_transformer.joblib')
df, test_spline = data_engineering('../data/testing_data.csv', train_spline)

In [21]:
y_test = df['Hydrogen_yield(kg)']
poly_features = ['Windspeed(m/s)' , 'GHI(W/m2)']
X_test_poly = df[poly_features]

# Pulling the trained models
poly_trans = joblib.load('../models/poly_trans.joblib')
poly_reg_model = joblib.load('../models/poly_reg_model.joblib')

X_test_poly_trans = poly_trans.transform(X_test_poly)

y_pred_poly = poly_reg_model.predict(X_test_poly_trans)

In [42]:
# Performance for the 2025 data
r2 = r2_score(y_test, y_pred_poly)
rmse = root_mean_squared_error(y_test, y_pred_poly)
n = X_test_poly_trans.shape[0]
p = X_test_poly_trans.shape[1]
adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))

print("2025 TEST DATA RESULTS (Polynomial Regression)")
print(f"RMSE:      {rmse:.2f} kg")
print(f"R² Score:  {r2:.4f}")
print(f"Adjusted R2 Score : {adjusted_r2:.4f}")

2025 TEST DATA RESULTS (Polynomial Regression)
RMSE:      19.65 kg
R² Score:  0.8814
Adjusted R2 Score : 0.8812


In [43]:
# Testing Random Forest over unseen data
features = ['GHI(W/m2)', 'Windspeed(m/s)', 'Stored Energy(MWh)', 'Mon',
            'Day', 'spline_hr_1','spline_hr_2', 'spline_hr_3',
            'spline_hr_4', 'Windspeed_mean_3h','Windspeed_std_3h',
            'GHI_mean_3h', 'Windspeed_lag_1hr','GHI_lag_1hr'
            ]
# Loading the model we saved from our script
rf_model = joblib.load('../models/random_forest_regressor.joblib')

X_test_rf = df[features]
y_pred_rf = rf_model.predict(X_test_rf)

rmse_rf = root_mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)
n_rf = X_test_rf.shape[0]
p_rf = X_test_rf.shape[1]
# Checking adjusted r2 to see if any of the features used are noise
adjusted_r2_rf = 1 - ((1 - r2_rf) * (n_rf - 1) / (n_rf - p_rf - 1))

print('2025 TEST DATA RESULTS (Random Forest)')
print(f"RMSE:      {rmse_rf:.2f} kg")
print(f"R² Score:  {r2_rf:.4f}")
print(f"Adjusted R2 Score : {adjusted_r2_rf:.4f}")

2025 TEST DATA RESULTS (Random Forest)
RMSE:      4.51 kg
R² Score:  0.9938
Adjusted R2 Score : 0.9937
